# Used Car Price Prediction: KNN

### Dataset

It is a comma separated file and there are 14 columns in the dataset.

- Location - The location in which the car is being sold or is available for purchase.
- Year - The year or edition of the model.
- KM_Driven - The total kilometers are driven in the car by the previous owner(s) in '000 KM.
- Fuel_Type - The type of fuel used by the car. (Petrol, Diesel, Electric, CNG, LPG)
- Transmission - The type of transmission used by the car. (Automatic / Manual)
- Owner_Type - First, Second, Third, or Fourth & Above
- Mileage - The standard mileage offered by the car company in kmpl or km/kg
- Engine - The displacement volume of the engine in CC.
- Power - The maximum power of the engine in bhp.
- Seats - The number of seats in the car.
- Price - The price of the car (target).

### Load Dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn

In [3]:
cars_df = pd.read_csv('final_cars_maruti.csv')

In [4]:
cars_df.sample(5)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,Age,Model,Mileage,Power,KM_Driven
22,Chennai,Petrol,Manual,Second,5,1.60,10,alto,19.70,46.30,94
268,Kochi,Petrol,Manual,First,5,6.21,5,swift,20.85,83.14,24
114,Ahmedabad,Diesel,Manual,First,7,7.75,3,ertiga,25.47,88.50,63
48,Mumbai,Petrol,Manual,First,7,8.25,2,ertiga,16.02,93.70,18
835,Bangalore,Petrol,Manual,Second,5,2.85,14,swift,20.40,81.80,80


In [5]:
cars_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Location      1010 non-null   object 
 1   Fuel_Type     1010 non-null   object 
 2   Transmission  1010 non-null   object 
 3   Owner_Type    1010 non-null   object 
 4   Seats         1010 non-null   int64  
 5   Price         1010 non-null   float64
 6   Age           1010 non-null   int64  
 7   Model         1010 non-null   object 
 8   Mileage       1010 non-null   float64
 9   Power         1010 non-null   float64
 10  KM_Driven     1010 non-null   int64  
dtypes: float64(3), int64(3), object(5)
memory usage: 86.9+ KB


In [6]:
cars_df.sample(10)

,Location,Fuel_Type,Transmission,Owner_Type,Seats,Price,Age,Model,Mileage,Power,KM_Driven
878,Bangalore,Petrol,Manual,Second,7,6.75,6,ertiga,16.02,93.70,68
73,Jaipur,Diesel,Manual,First,5,3.00,9,ritz,23.20,73.94,91
304,Coimbatore,Petrol,Automatic,First,5,5.23,3,wagon,22.50,67.00,37
21,Hyderabad,Petrol,Manual,First,5,2.60,6,alto,24.07,67.10,88
289,Kolkata,Diesel,Manual,First,5,4.35,6,swift,23.40,74.00,32
612,Jaipur,Petrol,Manual,First,5,2.75,5,alto,22.74,47.30,50
425,Kochi,Petrol,Manual,First,5,5.89,4,swift,16.10,85.00,19
132,Coimbatore,Petrol,Manual,Second,5,6.21,6,swift,18.60,85.80,33
954,Hyderabad,Diesel,Manual,First,5,6.85,5,swift,26.59,74.00,59
512,Chennai,Diesel,Manual,First,7,7.00,8,ertiga,20.77,88.80,40


### Feature Set Selection

In [7]:
cars_df.columns

Index(['Location', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Seats', 'Price',
       'Age', 'Model', 'Mileage', 'Power', 'KM_Driven'],
      dtype='object')

In [54]:
x_features = ['KM_Driven', 'Fuel_Type', 'Age',
              'Transmission', 'Owner_Type', 'Model']

In [55]:
cat_vars = ['Fuel_Type',
                'Transmission', 'Owner_Type',
                'Model']

In [56]:
num_vars = list(set(x_features) - set(cat_vars))

In [57]:
num_vars

['KM_Driven', 'Age']

In [58]:
cars_df[x_features].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   KM_Driven     1010 non-null   int64 
 1   Fuel_Type     1010 non-null   object
 2   Age           1010 non-null   int64 
 3   Transmission  1010 non-null   object
 4   Owner_Type    1010 non-null   object
 5   Model         1010 non-null   object
dtypes: int64(2), object(4)
memory usage: 47.5+ KB


### Need for Data Transformation

1. Data imputation for Seats Column
    - Mean imputation
2. Categorical Encoding for categorical columns
    - OHE Encoding
3. Data scaling
    - Standard scaling

### Setting X and y variables

In [59]:
X = cars_df[x_features]
y = cars_df['Price']

### Data Splitting

In [60]:
from sklearn.model_selection import train_test_split

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    train_size = 0.8,
                                                    random_state = 80)

In [62]:
X_train.shape

(808, 6)

In [63]:
X_test.shape

(202, 6)

### Data Imputation

In [64]:
from sklearn.impute import SimpleImputer

In [65]:
imputed_num_vars = ['Seats']

In [66]:
imputed_num_vars

['Seats']

In [67]:
non_imputed_num_vars = list(set(num_vars) - set(imputed_num_vars))

In [68]:
non_imputed_num_vars

['KM_Driven', 'Age']

In [69]:
mean_imputer = SimpleImputer(strategy='mean')

### Encode Categorical Variables

In [70]:
from sklearn.preprocessing import OneHotEncoder

In [71]:
ohe_encoder = OneHotEncoder(handle_unknown='ignore')

### Scaling numerical vars

In [72]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

### Creating Pipelines

In [73]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [74]:
imputed_num_transformer = Pipeline( steps = [
        ('imputation', mean_imputer),
        ('scaler', scaler)])

In [75]:
non_imputed_num_transformer = Pipeline( steps = [('scaler', scaler)])

In [76]:
cat_transformer = Pipeline( steps = [('ohencoder', ohe_encoder)])

In [77]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num_not_imputed', non_imputed_num_transformer, non_imputed_num_vars),
        ('catvars', cat_transformer, cat_vars)])

### KNN (K-Nearest Neighbor)


In [78]:
from sklearn.neighbors import KNeighborsRegressor

In [79]:
#knn = KNeighborsRegressor(n_neighbors=20)
knn = KNeighborsRegressor(n_neighbors=20, weights='distance')

In [80]:
knn_v1 = Pipeline(steps=[('preprocessor', preprocessor),
                          ('knn', knn)])

In [81]:
knn_v1.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_not_imputed',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['KM_Driven', 'Age']),
                                                 ('catvars',
                                                  Pipeline(steps=[('ohencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Fuel_Type', 'Transmission',
                                                   'Owner_Type', 'Model'])])),
                ('knn',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

In [82]:
from sklearn import set_config
set_config(display='diagram')

In [83]:
knn_v1

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_not_imputed',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['KM_Driven', 'Age']),
                                                 ('catvars',
                                                  Pipeline(steps=[('ohencoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Fuel_Type', 'Transmission',
                                                   'Owner_Type', 'Model'])])),
                ('knn',
                 KNeighborsRegressor(n_neighbors=20, weights='distance'))])

### Predict on test set

In [84]:
y_pred = knn_v1.predict(X_test)

### K Fold Cross Validation

In [85]:
from sklearn.model_selection import cross_val_score

In [86]:
scores = cross_val_score( knn_v1,
                          X_train,
                          y_train,
                          cv = 10,
                          scoring = 'r2')

In [87]:
scores

array([0.83719904, 0.85503347, 0.83046009, 0.85093421, 0.84064089,
       0.8282538 , 0.79423513, 0.87336603, 0.87520529, 0.84934506])

In [88]:
scores.mean()

0.8434672999233992

In [89]:
scores.std()

0.02235405343569982

In [90]:
from joblib import dump

In [91]:
dump(knn_v1, "cars_maruti.pkl")

['cars_maruti.pkl']

In [92]:
import sklearn
sklearn.__version__

'1.6.1'